# Rescue Vision — YOLOv8 Fine-Tuning (Kaggle)
**18-class accident detection model**

### Before running:
1. Enable GPU: Settings → Accelerator → **GPU T4 x2** (or P100)
2. Add dataset: Add Data → search `sushanthip/resq-vision-dataset`
3. Run all cells top to bottom

In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install ultralytics -q

In [ ]:
# ── Cell 2: CONFIG ─────────────────────────────────────────────────────────────

DATASET_PATH = '/kaggle/input/resq-vision-dataset'   # auto-mounted by Kaggle
OUTPUT_PATH  = '/kaggle/working/runs/rescue_vision_v2'

# Set to a .pt path if resuming, else None to start from yolov8n.pt
RESUME_WEIGHTS = None  # e.g. '/kaggle/input/my-weights/best.pt'

# Training hyperparameters
EPOCHS   = 100
BATCH    = 16   # T4 handles 16 at 640px; use 8 if OOM
IMGSZ    = 640
PATIENCE = 20
WORKERS  = 2    # Kaggle recommends 2

In [ ]:
# ── Cell 3: Verify GPU ─────────────────────────────────────────────────────────
!nvidia-smi | head -20

In [ ]:
# ── Cell 4: Inspect dataset structure ─────────────────────────────────────────
import os

for root, dirs, files in os.walk(DATASET_PATH):
    depth = root.replace(DATASET_PATH, '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root)}/')
    if depth == 2:
        print(f'{indent}  ({len(files)} files)')

In [ ]:
# ── Cell 5: Write dataset.yaml with correct Kaggle paths ──────────────────────
import yaml, os

# Auto-detect if dataset has a nested subfolder (e.g. merged_dataset/)
subdirs = [d for d in os.listdir(DATASET_PATH)
           if os.path.isdir(os.path.join(DATASET_PATH, d))]
print('Top-level dirs in dataset:', subdirs)

# Pick the folder that contains train/valid splits
dataset_root = DATASET_PATH
for d in subdirs:
    candidate = os.path.join(DATASET_PATH, d)
    if os.path.exists(os.path.join(candidate, 'train')):
        dataset_root = candidate
        break

print(f'Using dataset root: {dataset_root}')

cfg = {
    'path' : dataset_root,
    'train': 'train/images',
    'val'  : 'valid/images',
    'nc'   : 18,
    'names': [
        'bicycle', 'motorcycle', 'auto-rickshaw', 'car', 'bus', 'truck', 'person',
        'bicycle-bicycle_collision', 'bicycle-person_collision',
        'bicycle-object_collision', 'motorcycle-car_collision',
        'motorcycle-person_collision', 'car-car_collision',
        'car-person_collision', 'car-object_collision',
        'car-bicycle_collision', 'bus-collision', 'truck-collision'
    ]
}

yaml_path = '/kaggle/working/dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f'dataset.yaml written to {yaml_path}')
print(open(yaml_path).read())

In [ ]:
# ── Cell 6: Verify split counts ───────────────────────────────────────────────
import os

for split in ['train', 'valid']:
    img_dir = os.path.join(dataset_root, split, 'images')
    lbl_dir = os.path.join(dataset_root, split, 'labels')
    imgs = len(os.listdir(img_dir)) if os.path.exists(img_dir) else 0
    lbls = len(os.listdir(lbl_dir)) if os.path.exists(lbl_dir) else 0
    print(f'{split}: {imgs} images, {lbls} labels')

In [ ]:
# ── Cell 7: Train ──────────────────────────────────────────────────────────────
from ultralytics import YOLO

model_path = RESUME_WEIGHTS if RESUME_WEIGHTS else 'yolov8n.pt'
model = YOLO(model_path)

results = model.train(
    data    = '/kaggle/working/dataset.yaml',
    epochs  = EPOCHS,
    batch   = BATCH,
    imgsz   = IMGSZ,
    patience= PATIENCE,
    workers = WORKERS,
    device  = 0,
    project = '/kaggle/working/runs',
    name    = 'rescue_vision_v2',
    optimizer    = 'AdamW',
    cos_lr       = True,
    lr0          = 0.001,
    lrf          = 0.01,
    momentum     = 0.937,
    weight_decay = 0.0005,
    warmup_epochs    = 3,
    warmup_momentum  = 0.8,
    warmup_bias_lr   = 0.1,
    box     = 7.5,
    cls     = 0.7,
    dfl     = 1.5,
    hsv_h   = 0.015,
    hsv_s   = 0.7,
    hsv_v   = 0.4,
    degrees = 5.0,
    translate    = 0.1,
    scale        = 0.5,
    shear        = 2.0,
    perspective  = 0.0005,
    fliplr       = 0.5,
    mosaic       = 1.0,
    mixup        = 0.15,
    copy_paste   = 0.1,
    close_mosaic = 10,
    save_period  = 10,
    amp          = True,
    plots        = True,
    pretrained   = True,
    seed         = 0,
)

In [ ]:
# ── Cell 8: Validation summary ─────────────────────────────────────────────────
from ultralytics import YOLO

best = YOLO('/kaggle/working/runs/rescue_vision_v2/weights/best.pt')
metrics = best.val(data='/kaggle/working/dataset.yaml', device=0)
print('mAP50    :', metrics.box.map50)
print('mAP50-95 :', metrics.box.map)

In [ ]:
# ── Cell 9: Package weights for download ──────────────────────────────────────
import shutil, os

weights_dir = '/kaggle/working/runs/rescue_vision_v2/weights'
out_zip     = '/kaggle/working/rescue_vision_v2_weights'

shutil.make_archive(out_zip, 'zip', weights_dir)
print(f'Download: {out_zip}.zip')
print('Files:', os.listdir(weights_dir))